# 노코드 AI 서비스 구현
- 코딩 에이전트로 파이썬 업무 자동화

In [1]:
1 + 1

2

In [2]:
# 기본 파이썬 패키지로 PDF 내용 확인

"""
2026년 예산 사업설명자료.pdf 파일의 내용이 어떤 것이 있는지 확인해주는 파이썬 코드를 작성하라.
"""

import re
import zlib
from pathlib import Path

PDF_PATH = Path("./2026년 예산 사업설명자료.pdf")


def read_pdf_bytes(path):
    if not path.exists():
        current_dir = Path.cwd()
        print(current_dir)
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")

    return path.read_bytes()


def get_pdf_version(data):
    match = re.search(rb"%PDF-(\d\.\d)", data[:1024])
    if match:
        return match.group(1).decode("ascii", errors="ignore")
    return "PDF 헤더를 찾을 수 없습니다."


def estimate_page_count(data):
    page_matches = re.findall(rb"/Type\s*/Page\b", data)
    pages_matches = re.findall(rb"/Type\s*/Pages\b", data)

    estimated = len(page_matches) - len(pages_matches)

    if estimated < 0:
        estimated = len(page_matches)

    return estimated


def decode_pdf_string(raw):
    result = []
    i = 0

    while i < len(raw):
        char = raw[i]

        if char == 92:
            i += 1
            if i >= len(raw):
                break

            escaped = raw[i]

            escape_map = {
                ord("n"): "\n",
                ord("r"): "\r",
                ord("t"): "\t",
                ord("b"): "\b",
                ord("f"): "\f",
                ord("("): "(",
                ord(")"): ")",
                ord("\\"): "\\",
            }

            if escaped in escape_map:
                result.append(escape_map[escaped])
            elif 48 <= escaped <= 55:
                octal_digits = bytes([escaped])
                for _ in range(2):
                    if i + 1 < len(raw) and 48 <= raw[i + 1] <= 55:
                        i += 1
                        octal_digits += bytes([raw[i]])
                    else:
                        break

                try:
                    result.append(chr(int(octal_digits, 8)))
                except ValueError:
                    pass
            else:
                result.append(chr(escaped))
        else:
            result.append(chr(char))

        i += 1

    return "".join(result)


def extract_literal_strings(data, limit=200):
    strings = []
    pattern = re.compile(rb"\((?:\\.|[^\\()]){2,}\)")

    for match in pattern.finditer(data):
        raw = match.group(0)[1:-1]
        text = decode_pdf_string(raw)

        cleaned = " ".join(text.split())

        if len(cleaned) >= 2:
            strings.append(cleaned)

        if len(strings) >= limit:
            break

    return strings


def extract_metadata(data):
    metadata_keys = [
        "Title",
        "Author",
        "Subject",
        "Creator",
        "Producer",
        "CreationDate",
        "ModDate",
    ]

    metadata = {}

    for key in metadata_keys:
        pattern = rb"/" + key.encode("ascii") + rb"\s*\((.*?)\)"
        match = re.search(pattern, data, re.DOTALL)

        if match:
            metadata[key] = decode_pdf_string(match.group(1))

    return metadata


def extract_streams(data, max_streams=30):
    streams = []
    pattern = re.compile(rb"stream\r?\n(.*?)\r?\nendstream", re.DOTALL)

    for index, match in enumerate(pattern.finditer(data), start=1):
        stream_data = match.group(1)

        info = {
            "index": index,
            "raw_size": len(stream_data),
            "decompressed": False,
            "preview": "",
        }

        try:
            decompressed = zlib.decompress(stream_data)
            info["decompressed"] = True

            preview = decompressed[:1000].decode("utf-8", errors="ignore")
            info["preview"] = " ".join(preview.split())
        except Exception:
            preview = stream_data[:500].decode("latin-1", errors="ignore")
            info["preview"] = " ".join(preview.split())

        streams.append(info)

        if len(streams) >= max_streams:
            break

    return streams


def search_keywords(data, keywords):
    results = {}

    text_utf8 = data.decode("utf-8", errors="ignore")
    text_latin1 = data.decode("latin-1", errors="ignore")

    for keyword in keywords:
        count = text_utf8.count(keyword)

        if count == 0:
            count = text_latin1.count(keyword)

        results[keyword] = count

    return results


def main():
    data = read_pdf_bytes(PDF_PATH)

    print("PDF 기본 정보")
    print("=" * 60)
    print(f"파일 경로: {PDF_PATH}")
    print(f"파일 크기: {len(data):,} bytes")
    print(f"PDF 버전: {get_pdf_version(data)}")
    print(f"페이지 수 추정: {estimate_page_count(data)}")

    metadata = extract_metadata(data)

    print("\n메타데이터 추정")
    print("=" * 60)

    if metadata:
        for key, value in metadata.items():
            print(f"{key}: {value}")
    else:
        print("기본 방식으로 확인 가능한 메타데이터가 없습니다.")

    strings = extract_literal_strings(data, limit=100)

    print("\nPDF 내부 문자열 일부")
    print("=" * 60)

    if strings:
        for index, text in enumerate(strings[:50], start=1):
            print(f"{index:02d}. {text}")
    else:
        print("기본 방식으로 추출 가능한 문자열이 거의 없습니다.")

    keywords = [
        "예산",
        "사업",
        "설명",
        "2026",
        "회계",
        "세출",
        "세입",
        "국고",
        "지원",
        "성과",
    ]

    keyword_results = search_keywords(data, keywords)

    print("\n키워드 검색 결과")
    print("=" * 60)

    for keyword, count in keyword_results.items():
        print(f"{keyword}: {count}회")

    streams = extract_streams(data, max_streams=20)

    print("\nPDF 스트림 분석")
    print("=" * 60)
    print(f"확인한 스트림 수: {len(streams)}")

    for stream in streams[:10]:
        print(f"\n--- Stream {stream['index']} ---")
        print(f"원본 크기: {stream['raw_size']:,} bytes")
        print(f"압축 해제 여부: {stream['decompressed']}")

        if stream["preview"]:
            print("미리보기:")
            print(stream["preview"][:500])
        else:
            print("미리보기 없음")

    print("\n분석 완료")
    print("=" * 60)
    print("참고: PDF가 압축/인코딩/images 기반이면 기본 파이썬만으로 실제 본문 추출이 제한됩니다.")


if __name__ == "__main__":
    main()


PDF 기본 정보
파일 경로: 2026년 예산 사업설명자료.pdf
파일 크기: 29,291,332 bytes
PDF 버전: 1.4
페이지 수 추정: 2166

메타데이터 추정
Title: þÿ 2 0 2 6±D³Ä  À°  À¬ÅÅÁ$ºÇ¸Ì (ÖHÓÇtÉÀ  ¬ÂÜ 
Author: DS6
Creator: Hwp 2022 12.0.0.535
Producer: Hancom PDF 1.3.0.538
CreationDate: D:20260120085512+09'00'
ModDate: D:20260120085606+09'00'

PDF 내부 문자열 일부
01. ko-KR
02. D:20260120085533
03. D:20260120085533
04. D:20260120085533
05. D:20260120085533
06. D:20260120085533
07. D:20260120085533
08. D:20260120085533
09. D:20260120085533
10. D:20260120085533
11. D:20260120085533
12. D:20260120085533
13. D:20260120085533
14. D:20260120085533
15. D:20260120085533
16. D:20260120085533
17. D:20260120085533
18. D:20260120085533
19. D:20260120085533
20. D:20260120085533
21. D:20260120085533
22. D:20260120085533
23. D:20260120085533
24. D:20260120085533
25. D:20260120085533
26. D:20260120085533
27. D:20260120085533
28. D:20260120085533
29. D:20260120085533
30. D:20260120085533
31. D:20260120085533
32. D:20260120085533
33. D:20260120085533
34. D

In [3]:
# PyMuPDF 패키지 이용

import importlib.util

# 패키지 설치 여부 확인
if importlib.util.find_spec("fitz") is None:
    print("PyMuPDF가 설치되어 있지 않습니다.")
    print("아래 명령어를 새 셀에서 실행하세요:")
    print("!conda install -c conda-forge pymupdf -y")
else:
    print("PyMuPDF가 설치되어 있습니다.")

PyMuPDF가 설치되어 있습니다.


In [4]:
# PDF 기본 정보와 앞부분 내용 확인

from pathlib import Path
import importlib.util

PDF_PATH = Path("./2026년 예산 사업설명자료.pdf")

if importlib.util.find_spec("fitz") is None:
    print("PyMuPDF가 설치되어 있지 않습니다.")
    print("설치 명령어: !conda install -c conda-forge pymupdf -y")
else:
    import fitz

    if not PDF_PATH.exists():
        print(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH}")
    else:
        doc = fitz.open(PDF_PATH)

        print("PDF 기본 정보")
        print("=" * 60)
        print(f"파일 경로: {PDF_PATH}")
        print(f"페이지 수: {doc.page_count}")

        print("\nPDF 메타데이터")
        print("=" * 60)
        for key, value in doc.metadata.items():
            print(f"{key}: {value}")

        print("\nPDF 본문 미리보기")
        print("=" * 60)

        preview_pages = min(3, doc.page_count)

        for page_index in range(preview_pages):
            page = doc.load_page(page_index)
            text = page.get_text()

            print(f"\n--- {page_index + 1} 페이지 ---")
            if text.strip():
                print(text[:2000])
            else:
                print("이 페이지에서 추출된 텍스트가 없습니다.")

        doc.close()

PDF 기본 정보
파일 경로: 2026년 예산 사업설명자료.pdf
페이지 수: 2170

PDF 메타데이터
format: PDF 1.4
title: 
author: DS6
subject: 
keywords: 
creator: Hwp 2022 12.0.0.535
producer: Hancom PDF 1.3.0.538
creationDate: D:20260120085512+09'00'
modDate: D:20260120085606+09'00'
trapped: 
encryption: None

PDF 본문 미리보기

--- 1 페이지 ---
2026년도 예산 및 기금운용계획 사업설명자료(Ⅱ-1)
2026. 1.


--- 2 페이지 ---
이 페이지에서 추출된 텍스트가 없습니다.

--- 3 페이지 ---
- i -
목   차
Ⅰ. 일반회계···················································································· 1
 세
입·····································································································3
 세
출···································································································23
<대변인실>
1. 대변인실(총액) ·····································································································27
2. 대변인실···············································································································31
3. 국민안전방송운영및재난정책홍보····················

In [5]:
# PDF 전체 텍스트를 변수에 저장

from pathlib import Path
import fitz

PDF_PATH = Path("./2026년 예산 사업설명자료.pdf")

all_text = ""

doc = fitz.open(PDF_PATH)

for page_index in range(doc.page_count):
    page = doc.load_page(page_index)
    text = page.get_text()
    all_text += f"\n\n--- {page_index + 1} 페이지 ---\n"
    all_text += text

doc.close()

print(f"전체 텍스트 길이: {len(all_text):,} 글자")
print(all_text[:3000])

전체 텍스트 길이: 1,898,617 글자


--- 1 페이지 ---
2026년도 예산 및 기금운용계획 사업설명자료(Ⅱ-1)
2026. 1.


--- 2 페이지 ---


--- 3 페이지 ---
- i -
목   차
Ⅰ. 일반회계···················································································· 1
 세
입·····································································································3
 세
출···································································································23
<대변인실>
1. 대변인실(총액) ·····································································································27
2. 대변인실···············································································································31
3. 국민안전방송운영및재난정책홍보·······························································35
<기획조정실>
1. 기획조정실(총액) ·································································································43
2. 기관운영및예산지원관리(총액) ·······································································48
3. 기획조정실·················

In [6]:
# 특정 키워드 검색

keywords = ["예산", "사업", "국고", "지원", "성과", "세출", "세입"]

for keyword in keywords:
    count = all_text.count(keyword)
    print(f"{keyword}: {count}회")

예산: 6803회
사업: 13739회
국고: 424회
지원: 4799회
성과: 2103회
세출: 7회
세입: 58회


In [7]:
# '□' 아래에 있는 표만 추출

"""
PDF 파일 100~150 페이지에서 □이 있는 글자 아래의 표만 추출하는 코드를 줘.
주피터 노트북에서 실행할 거야.
"""

from pathlib import Path
import fitz
import pandas as pd

PDF_PATH = Path("./2026년 예산 사업설명자료.pdf")

results = []

doc = fitz.open(PDF_PATH)

for page_index in range(99, min(150, doc.page_count)):
    page = doc.load_page(page_index)

    # 페이지 안의 텍스트 블록 추출
    blocks = page.get_text("blocks")

    # '□'가 들어간 텍스트 블록 찾기
    square_blocks = []

    for block in blocks:
        x0, y0, x1, y1, text, block_no, block_type = block

        if "□" in text:
            square_blocks.append({
                "x0": x0,
                "y0": y0,
                "x1": x1,
                "y1": y1,
                "text": text.strip()
            })

    if not square_blocks:
        continue

    # 페이지의 tables 찾기
    try:
        tables = page.find_tables()
    except Exception as e:
        print(f"{page_index + 1}페이지에서 tables 찾기 실패: {e}")
        continue

    if not tables.tables:
        continue

    for square_block in square_blocks:
        square_y_bottom = square_block["y1"]

        # 현재 □ 블록 아래에 있는 표만 선택
        candidate_tables = []

        for table_index, table in enumerate(tables.tables):
            table_x0, table_y0, table_x1, table_y1 = table.bbox

            if table_y0 > square_y_bottom:
                candidate_tables.append({
                    "table_index": table_index,
                    "bbox": table.bbox,
                    "table": table,
                    "distance": table_y0 - square_y_bottom
                })

        # □ 바로 아래에 있는 가장 가까운 표만 추출
        if candidate_tables:
            nearest_table = sorted(candidate_tables, key=lambda x: x["distance"])[0]
            extracted = nearest_table["table"].extract()

            results.append({
                "page": page_index + 1,
                "square_text": square_block["text"],
                "table_index": nearest_table["table_index"],
                "table_bbox": nearest_table["bbox"],
                "table_data": extracted
            })

doc.close()

print(f"추출된 tables 개수: {len(results)}")

추출된 표 개수: 76


In [8]:
# 추출 결과 일부만 확인

MAX_DISPLAY_TABLES = 5  # 출력할 tables 개수

print(f"전체 추출된 tables 개수: {len(results)}")
print(f"화면에 출력할 tables 개수: {min(MAX_DISPLAY_TABLES, len(results))}")

for i, item in enumerate(results[:MAX_DISPLAY_TABLES], start=1):
    print("=" * 80)
    print(f"{i}번째 tables")
    print(f"페이지: {item['page']}")
    print(f"□ 포함 문장:")
    print(item["square_text"])
    print("\ntables 내용:")

    df = pd.DataFrame(item["table_data"])
    display(df)

전체 추출된 표 개수: 76
화면에 출력할 표 개수: 5
1번째 표
페이지: 100
□ 포함 문장:
□사업영향, 산출물 성과지표 등

표 내용:


,0,1
0,2022,- 성과관리 시행계획 수립\n- 조직 및 개인성과 평가 시행\n- 혁신역량교육 운영...
1,2023,- 성과관리 시행계획 수립\n- 조직 및 개인성과 평가 시행\n- 혁신역량교육 운영...
2,2024,- 성과관리 시행계획 수립\n- 조직 및 개인성과 평가 시행\n- 혁신역량교육 운영...
3,2025,- 성과관리 시행계획 수립\n- 조직 및 개인성과 평가 시행\n- 혁신역량교육 운영...


2번째 표
페이지: 102
□ 포함 문장:
□부처 결산내역

표 내용:


,0,1,2,3,4,5,6,7,8
0,연도,예산액,None,None,예산현액\n(A),집행액\n(B),집행률\n(B/A),다음연도\n이월액,불용액
1,None,본예산,추경\n증감액,추경,None,None,None,None,None
2,2022,330,△12,318,318,279,87.7,-,39
3,2023,331,-,331,350,333,95.1,-,18
4,2024,"3,369",-,"3,369","3,369","2,913",86.5,19,437
5,2025,"3,417",-,"3,417","3,436","3,217",93.6,-,219


3번째 표
페이지: 102
□ 포함 문장:
□2022~2025년 결산 주요사항

표 내용:


,0,1
0,2022,- 불용액(39백만원)은 인건비 및 집행잔액
1,2023,- 이·전용 등(30백만원)은 국정과제 분야별 추진상황 점검 등을 위한 회의 개최 ...
2,2024,- 이·전용 등(64백만원)은 청년인턴 채용 인원 증가 및 부내 행사 등 임차료 마...
3,2025,"- 이·전용 등(23백만원)은 청년인턴 기본교육 및 연구소모임, 과장급 역량교육 등..."


4번째 표
페이지: 102
□ 포함 문장:
□2025년 이․전용 등 세부내역

표 내용:


,0,1,2,3,4,5,6
0,구분\n(날짜),~에서,None,금액,~으로,None,이․전용 등 사유
1,None,세부사업 명\n(사업코드),목-세목\n코드,None,세부사업 명\n(사업코드),목-세목\n코드,None
2,세목조정\n(2025.2.3.),성과관리 및\n행정효율성제고\n(7033-300),210-01,20,성과관리 및\n행정효율성제고\n(7033-300),210-07,"청년인턴 기본교육 교육장\n및 버스 임차, 연구소모임\n운영, 청년인턴 채용서류\n..."
3,세목조정\n(2025.2.10.),성과관리 및\n행정효율성제고\n(7033-300),210-01,3,성과관리 및\n행정효율성제고\n(7033-300),210-07,과장급 역량강화 교육\n관련 이동수단 및\n교육장소 임차


5번째 표
페이지: 103
□ 포함 문장:
□사업 코드 정보

표 내용:


,0,1,2,3,4,5,6
0,구분,회계,소관,실국(기관),계정,분야,부문
1,코드,일반회계,행정안전부,기획조정실,,010,016
2,명칭,None,None,None,None,일반·지방행정,일반행정


In [9]:
# 추출한 표를 1개의 엑셀 파일에 저장 (100~150페이지)

from pathlib import Path
import pandas as pd

output_path = Path("./extracted_tables_page100_150.xlsx")

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for i, item in enumerate(results, start=1):
        table_data = item["table_data"]

        if not table_data:
            continue

        df = pd.DataFrame(table_data)
        sheet_name = f"p{item['page']}_t{i}"
        # 시트 이름은 최대 31자
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False, header=False)

print(f"엑셀 파일 저장 완료: {output_path.resolve()}")
print(f"총 {len(results)}개의 표가 저장됨")

엑셀 파일 저장 완료: C:\Users\user\Documents\Study\AI-Champion\python_exercise\extracted_tables_page100_150.xlsx
총 76개의 표가 저장됨


In [10]:
"""
엑셀 파일을 100개 가상으로 만들어줘.
10행 5열 짜리 컬럼이 있는 엑셀 파일을 만들고 각 파일을 1개의 폴더에 모두 넣어주는 코드.
"""

# Jupyter 노트북용: 한 폴더에 100개의 엑셀 파일 생성 (각 파일: 10행 x 5열)

import os

def ensure_package(pkg_name, import_name=None):
    if import_name is None:
        import_name = pkg_name
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])

# 필요한 패키지 설치
ensure_package("pandas")
ensure_package("openpyxl")
ensure_package("numpy")

import pandas as pd
import numpy as np

folder = "fake_excels"  # 생성할 폴더 이름 (현재 작업 디렉터리 하위)
os.makedirs(folder, exist_ok=True)

np.random.seed(42)  # 재현성 옵션 (원하면 제거)
columns = [f"Col{i}" for i in range(1, 6)]

for i in range(1, 101):
    arr = np.random.randint(0, 1000, size=(10, 5))  # 10행 5열 정수 데이터
    df = pd.DataFrame(arr, columns=columns)
    filename = f"data_{i:03d}.xlsx"
    filepath = os.path.join(folder, filename)
    df.to_excel(filepath, index=False, engine="openpyxl")

print(f"완료: '{folder}' 폴더에 100개의 엑셀 파일을 생성했습니다.")


완료: 'fake_excels' 폴더에 100개의 엑셀 파일을 생성했습니다.


In [11]:
"""
fake_excels 디렉토리 내의 엑셀 파일 구조가 똑같다.
100개의 엑셀 파일을 위아래로 통합해서 단일 파일로 만든 후 나머지 99개를 삭제하는 코드를 작성하라.
"""

# 필요 패키지 자동 설치(없으면 설치)
def ensure_package(pkg, import_name=None):
    if import_name is None:
        import_name = pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
ensure_package("pandas")
ensure_package("openpyxl")  # openpyxl는 pandas to_excel용 엔진

import os, glob
import pandas as pd

# 설정
folder = "fake_excels"  # 작업할 폴더 (현재 작업 디렉터리 기준)
output_name = "merged_data.xlsx"  # 병합 결과 파일명 (같은 폴더에 생성)
output_path = os.path.join(folder, output_name)

# 안전 옵션: 실제로 파일을 삭제하려면 아래 값을 True로 변경하세요.
# 기본은 False(삭제하지 않음)입니다.
# PERFORM_DELETE = False
PERFORM_DELETE =True

# 1) 파일 목록 수집
pattern = os.path.join(folder, "*.xlsx")
files = sorted(glob.glob(pattern))

if not files:
    print(f"'{folder}' 폴더에 .xlsx 파일이 없습니다. 경로와 확장자를 확인하세요.")
else:
    print(f"발견된 파일 개수: {len(files)}")
    # 2) 각 파일 읽기
    dfs = []
    first_cols = None
    error_files = []
    for f in files:
        # skip the eventual existing merged output if present in the folder
        if os.path.abspath(f) == os.path.abspath(output_path):
            print(f"스킵(이미 병합 파일): {f}")
            continue
        try:
            df = pd.read_excel(f, engine="openpyxl")
            # 첫 파일의 컬럼 구조를 기억
            if first_cols is None:
                first_cols = list(df.columns)
            else:
                # 컬럼이 다르면 경고 후 맞춰서 정렬(없는 컬럼은 NaN)
                if list(df.columns) != first_cols:
                    print(f"경고: 컬럼 구조 불일치({os.path.basename(f)}). 첫 파일 컬럼에 맞춰 재정렬/합침.")
                    df = df.reindex(columns=first_cols)
            dfs.append(df)
        except Exception as e:
            print(f"파일 읽기 실패: {f} - {e}")
            error_files.append(f)

    if not dfs:
        print("읽힌 데이터프레임이 없습니다. 병합을 중단합니다.")
    else:
        # 3) 병합
        combined = pd.concat(dfs, ignore_index=True)
        # 4) 저장
        try:
            combined.to_excel(output_path, index=False, engine="openpyxl")
            print(f"병합 완료: {output_path} (행 수: {len(combined)}, 열 수: {len(combined.columns)})")
        except Exception as e:
            print(f"병합 저장 실패: {e}")
            raise

        # 5) 원본 삭제 (옵션)
        if PERFORM_DELETE:
            deleted = []
            failed = []
            for f in files:
                # 안전하게 병합본은 삭제 대상에서 제외
                if os.path.abspath(f) == os.path.abspath(output_path):
                    continue
                try:
                    os.remove(f)
                    deleted.append(f)
                except Exception as e:
                    failed.append((f, str(e)))
            print(f"삭제 완료한 파일 수: {len(deleted)}")
            if failed:
                print("삭제 실패 목록:")
                for ff, err in failed:
                    print(f" - {ff}: {err}")
        else:
            print("현재는 PERFORM_DELETE=False 입니다. 원본 파일은 삭제되지 않았습니다.")
            print("원본 파일을 삭제하려면 코드 상단의 PERFORM_DELETE를 True로 변경하세요.")
        if error_files:
            print("읽기 실패 파일이 존재합니다:")
            for ef in error_files:
                print(" -", ef)

발견된 파일 개수: 100
병합 완료: fake_excels\merged_data.xlsx (행 수: 1000, 열 수: 5)
삭제 완료한 파일 수: 100


In [13]:
"""
디렉토리 내의 '행정안전부_지역별(행정동) 성별 연령별 주민등록 인구수_20260531.csv' 이 데이터가 어떻게 생겼는지 보여주는 코드를 보여줘.
주피터노트북에서 실행하고 있다.
"""

from pathlib import Path
import sys
import subprocess

# 필요 라이브러리 설치(없을 경우)
def ensure_package(pkg, import_name=None):
    if import_name is None:
        import_name = pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
ensure_package("pandas")
ensure_package("numpy")
from IPython.display import display
import pandas as pd
import numpy as np

# 파일 경로 (필요하면 수정)
csv_path = Path(r"c:\Users\user\Documents\Study\AI-Champion\python_exercise\행정안전부_지역별(행정동) 성별 연령별 주민등록 인구수_20260531.csv")

if not csv_path.exists():
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {csv_path}")

print("파일:", csv_path)
print("파일 크기 (bytes):", csv_path.stat().st_size)

# 인코딩/구분자 시도 함수
def try_read_preview(path, encodings=None, sep_candidates=[",", None], nrows=20):
    if encodings is None:
        encodings = ["utf-8", "cp949", "euc-kr", "latin1"]
    last_error = None
    for enc in encodings:
        for sep in sep_candidates:
            try:
                # engine='python'이면 sep=None(탐지) 허용
                if sep is None:
                    df = pd.read_csv(path, encoding=enc, engine="python", nrows=nrows)
                else:
                    df = pd.read_csv(path, encoding=enc, sep=sep, nrows=nrows)
                return df, enc, sep
            except Exception as e:
                last_error = e
    # 실패 시 마지막 에러 전달
    raise last_error

# 미리보기로 읽기
try:
    preview_df, used_encoding, used_sep = try_read_preview(csv_path, nrows=50)
    print(f"성공: 인코딩={used_encoding!r}, sep={used_sep!r}")
except Exception as e:
    raise RuntimeError(f"파일 미리보기 읽기 실패: {e}")

# 기본 정보 출력
print("\n===== 컬럼(헤더) =====")
display(list(preview_df.columns))

print("\n===== 미리보기 (상위 10행) =====")
display(preview_df.head(10))

print("\n===== 미리보기 (하위 5행) =====")
display(preview_df.tail(5))

# 요약 정보 함수
def summarize_df(df, max_cat_values=10):
    info = {}
    info['shape'] = df.shape
    info['dtypes'] = df.dtypes
    # 결측치
    missing = df.isna().sum()
    missing_pct = (missing / len(df)) * 100
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_missing_count": df.notna().sum(),
        "missing_count": missing,
        "missing_pct": missing_pct.round(2)
    }))
    # 숫자형 통계
    num_df = df.select_dtypes(include=[np.number])
    if not num_df.empty:
        print("\n===== 숫자형 변수 요약 =====")
        display(num_df.describe().T)
    else:
        print("\n숫자형 컬럼이 없습니다.")
    # 범주형(또는 object) 상위값
    print("\n===== 범주형(또는 object) 컬럼 상위값 (최대 {}개) =====".format(max_cat_values))
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if not cat_cols:
        print("범주형(문자열) 컬럼이 없습니다.")
    else:
        for c in cat_cols:
            vc = df[c].value_counts(dropna=False).head(max_cat_values)
            print(f"\n-- {c} (유니크 {df[c].nunique(dropna=False)}):")
            display(vc)

# 미리보기 데이터로 요약 (파일 전체와 비슷한 구조 확인 목적)
print("\n===== 미리보기 데이터 구조 요약 =====")
summarize_df(preview_df, max_cat_values=8)

# (선택) 전체 파일을 메모리에 불러올지 여부 결정
# 파일이 너무 크면 메모리 문제 발생 가능 → chunksize 사용 권장
LOAD_FULL = False  # 전체 로드하려면 True로 변경
if LOAD_FULL:
    # 전체 로드 예시 (인코딩/sep는 위에서 성공한 값 사용)
    print("\n전체 파일을 메모리에 로드합니다. 메모리 부족 가능성에 유의하세요.")
    df_full = pd.read_csv(csv_path, encoding=used_encoding, sep=used_sep, engine="python" if used_sep is None else "c", low_memory=False)
    print("전체 로드 완료. shape:", df_full.shape)
    # 전체 데이터 요약을 보려면 아래 호출
    summarize_df(df_full, max_cat_values=10)
else:
    # chunksize로 행 수 확인 예시 (인구 칼럼 자동 탐지/합계 로직 제거됨)
    print("\n전체 로드를 수행하지 않았습니다. 대신 chunksize로 전체 행수 확인 예시:")
    try:
        total_rows = 0
        for chunk in pd.read_csv(csv_path, encoding=used_encoding, sep=used_sep, chunksize=200000, engine="python" if used_sep is None else "c"):
            total_rows += len(chunk)
        print("추정 전체 행 수:", total_rows)
    except Exception as e:
        print("chunksize로 파일을 읽는 중 에러:", e)

파일: c:\Users\user\Documents\Study\AI-Champion\python_exercise\행정안전부_지역별(행정동) 성별 연령별 주민등록 인구수_20260531.csv
파일 크기 (bytes): 2563889
성공: 인코딩='cp949', sep=','

===== 컬럼(헤더) =====


['행정기관코드',
 '기준연월',
 '시도명',
 '시군구명',
 '읍면동명',
 '계',
 '남자',
 '여자',
 '0세남자',
 '1세남자',
 '2세남자',
 '3세남자',
 '4세남자',
 '5세남자',
 '6세남자',
 '7세남자',
 '8세남자',
 '9세남자',
 '10세남자',
 '11세남자',
 '12세남자',
 '13세남자',
 '14세남자',
 '15세남자',
 '16세남자',
 '17세남자',
 '18세남자',
 '19세남자',
 '20세남자',
 '21세남자',
 '22세남자',
 '23세남자',
 '24세남자',
 '25세남자',
 '26세남자',
 '27세남자',
 '28세남자',
 '29세남자',
 '30세남자',
 '31세남자',
 '32세남자',
 '33세남자',
 '34세남자',
 '35세남자',
 '36세남자',
 '37세남자',
 '38세남자',
 '39세남자',
 '40세남자',
 '41세남자',
 '42세남자',
 '43세남자',
 '44세남자',
 '45세남자',
 '46세남자',
 '47세남자',
 '48세남자',
 '49세남자',
 '50세남자',
 '51세남자',
 '52세남자',
 '53세남자',
 '54세남자',
 '55세남자',
 '56세남자',
 '57세남자',
 '58세남자',
 '59세남자',
 '60세남자',
 '61세남자',
 '62세남자',
 '63세남자',
 '64세남자',
 '65세남자',
 '66세남자',
 '67세남자',
 '68세남자',
 '69세남자',
 '70세남자',
 '71세남자',
 '72세남자',
 '73세남자',
 '74세남자',
 '75세남자',
 '76세남자',
 '77세남자',
 '78세남자',
 '79세남자',
 '80세남자',
 '81세남자',
 '82세남자',
 '83세남자',
 '84세남자',
 '85세남자',
 '86세남자',
 '87세남자',
 '88세남자',
 '89세남자',
 '90세남자',
 '91세남자',
 '92세남자',
 '93세남자',
 '94


===== 미리보기 (상위 10행) =====


,행정기관코드,기준연월,시도명,시군구명,읍면동명,계,남자,여자,0세남자,1세남자,...,101세여자,102세여자,103세여자,104세여자,105세여자,106세여자,107세여자,108세여자,109세여자,110세이상 여자
0,1111051500,2026-05-31,서울특별시,종로구,청운효자동,10719,4869,5850,17,17,...,0,0,0,0,0,0,0,0,0,0
1,1111053000,2026-05-31,서울특별시,종로구,사직동,8874,3909,4965,9,17,...,0,0,1,0,0,0,0,0,0,0
2,1111054000,2026-05-31,서울특별시,종로구,삼청동,2064,976,1088,4,4,...,0,0,0,0,0,0,0,0,0,0
3,1111055000,2026-05-31,서울특별시,종로구,부암동,8872,4137,4735,13,20,...,1,0,1,0,0,0,0,0,0,0
4,1111056000,2026-05-31,서울특별시,종로구,평창동,16959,7810,9149,33,42,...,3,2,1,1,0,0,0,0,0,0
5,1111057000,2026-05-31,서울특별시,종로구,무악동,7699,3540,4159,12,23,...,0,0,0,0,0,0,0,0,0,0
6,1111058000,2026-05-31,서울특별시,종로구,교남동,9470,4278,5192,25,21,...,0,0,0,0,0,0,0,0,0,0
7,1111060000,2026-05-31,서울특별시,종로구,가회동,3525,1636,1889,5,6,...,1,0,0,0,0,0,0,0,0,0
8,1111061500,2026-05-31,서울특별시,종로구,종로1.2.3.4가동,6651,4005,2646,8,8,...,0,0,1,0,0,0,0,0,0,0
9,1111063000,2026-05-31,서울특별시,종로구,종로5.6가동,5420,2780,2640,8,7,...,0,0,0,0,0,0,0,0,0,0



===== 미리보기 (하위 5행) =====


...


-- 시군구명 (유니크 4):


시군구명
종로구    17
용산구    16
중구     15
성동구     2
Name: count, dtype: int64


-- 읍면동명 (유니크 50):


읍면동명
청운효자동     1
원효로제2동    1
청구동       1
신당제5동     1
동화동       1
황학동       1
중림동       1
후암동       1
Name: count, dtype: int64


전체 로드를 수행하지 않았습니다. 대신 chunksize로 전체 행수 확인 예시:
추정 전체 행 수: 3618
